In [8]:
"""
Step 1: Data Preprocessing
============================
目标：
1. 读取原始Excel数据
2. 将GPL201/GPL570列重命名，合并为同一个数据集 GSE43777
3. 解析所有127列的列名，提取：数据集(dataset)、重症类型(severity: DHF/DSS)、患者序号
4. 生成一份"数据集元数据表"，汇总每个数据集在DHF/DSS下各有多少患者
5. 打印验证结果
"""

import pandas as pd
import re
import numpy as np

# -------------------------------------------------
# 1.1 读取原始数据
# -------------------------------------------------
print("=" * 60)
print("STEP 1.1: 读取原始数据")
print("=" * 60)

df_raw = pd.read_excel("Severe_dengue_DHF_DSS_Xinyang.xlsx", sheet_name="Sheet1")  # 改成你本地的文件路径
GENE_COL = "Gene_Symbol"

print(f"原始数据形状 (行=基因, 列=Gene_Symbol+患者): {df_raw.shape}")
print(f"基因总数: {df_raw.shape[0]}")
print(f"患者列总数 (不含Gene_Symbol): {df_raw.shape[1] - 1}")
print(f"是否有重复基因名: {df_raw[GENE_COL].duplicated().sum()} 个重复")
print()

# -------------------------------------------------
# 1.2 重命名 GPL201 / GPL570 -> GSE43777（同一研究，两种芯片平台）
# -------------------------------------------------
print("=" * 60)
print("STEP 1.2: 重命名 GPL201/GPL570 为 GSE43777")
print("=" * 60)

gpl201_cols = [c for c in df_raw.columns if c.startswith("GPL201_")]
gpl570_cols = [c for c in df_raw.columns if c.startswith("GPL570_")]

print(f"GPL201 列数: {len(gpl201_cols)} -> {gpl201_cols}")
print(f"GPL570 列数: {len(gpl570_cols)} -> {gpl570_cols}")

final_rename = {}
idx = 1
for c in gpl201_cols:
    final_rename[c] = f"GSE43777_DHFvsDF_{idx}"
    idx += 1
for c in gpl570_cols:
    final_rename[c] = f"GSE43777_DHFvsDF_{idx}"
    idx += 1

df_raw = df_raw.rename(columns=final_rename)

print()
print("重命名映射结果:")
for old, new in final_rename.items():
    print(f"  {old:30s} -> {new}")
print()
print(f"合并后 GSE43777 共有 {len(final_rename)} 名患者 (原GPL201={len(gpl201_cols)} + GPL570={len(gpl570_cols)})")

# -------------------------------------------------
# 1.3 解析所有列名 -> (数据集, 严重程度, 患者序号)
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 1.3: 解析全部患者列名")
print("=" * 60)

expr_cols = [c for c in df_raw.columns if c != GENE_COL]
print(f"待解析的患者列总数: {len(expr_cols)}")

col_meta_records = []
for col in expr_cols:
    m = re.match(r"^(GSE\d+)_([A-Za-z]+)vs([A-Za-z]+)_(\d+)$", col)
    if m:
        dataset, severity, control, patient_idx = m.groups()
        col_meta_records.append({
            "column": col,
            "dataset": dataset,
            "severity": severity,   # DHF or DSS
            "control": control,     # DF
            "patient_idx": int(patient_idx),
        })
    else:
        col_meta_records.append({
            "column": col, "dataset": "UNPARSED", "severity": "UNPARSED",
            "control": "UNPARSED", "patient_idx": None,
        })

col_meta = pd.DataFrame(col_meta_records)

n_unparsed = (col_meta["dataset"] == "UNPARSED").sum()
print(f"解析失败(UNPARSED)的列数: {n_unparsed}")
assert n_unparsed == 0, "存在无法解析的列名，需要人工检查！"
print("✅ 全部127列成功解析")
print()
print("解析结果预览（前5行 + 后5行）:")
print(col_meta.head(5).to_string(index=False))
print("...")
print(col_meta.tail(5).to_string(index=False))

# -------------------------------------------------
# 1.4 数据集元数据汇总表：每个数据集 x 严重程度 的患者数
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 1.4: 生成数据集元数据汇总表")
print("=" * 60)

dataset_summary = (
    col_meta.groupby(["dataset", "severity"])
    .size()
    .reset_index(name="n_patients")
    .sort_values(["dataset", "severity"])
    .reset_index(drop=True)
)

print(dataset_summary.to_string(index=False))
print()
print(f"数据集总数（去重）: {col_meta['dataset'].nunique()}")
print(f"数据集列表: {sorted(col_meta['dataset'].unique())}")
print()
print(f"验证：所有severity列patient数总和 = {dataset_summary['n_patients'].sum()} (应等于127)")
assert dataset_summary["n_patients"].sum() == 127, "患者总数对不上127，请检查！"
print("✅ 患者总数校验通过：127")

print()
print(f"验证：DHF患者总数 = {dataset_summary[dataset_summary['severity']=='DHF']['n_patients'].sum()}")
print(f"验证：DSS患者总数 = {dataset_summary[dataset_summary['severity']=='DSS']['n_patients'].sum()}")

# -------------------------------------------------
# 1.5 保存处理后的数据，供下一步(核心查询函数)使用
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 1.5: 保存中间结果")
print("=" * 60)

df_raw.to_pickle("df_processed.pkl")
col_meta.to_pickle("col_meta.pkl")
dataset_summary.to_csv("dataset_summary.csv", index=False)

print("已保存: df_processed.pkl (重命名后的完整表达矩阵)")
print("已保存: col_meta.pkl (127列的元数据: 数据集/严重程度/患者序号)")
print("已保存: dataset_summary.csv (数据集汇总表，可用Excel打开查看)")

STEP 1.1: 读取原始数据
原始数据形状 (行=基因, 列=Gene_Symbol+患者): (12746, 128)
基因总数: 12746
患者列总数 (不含Gene_Symbol): 127
是否有重复基因名: 0 个重复

STEP 1.2: 重命名 GPL201/GPL570 为 GSE43777
GPL201 列数: 3 -> ['GPL201_DHF_1_log2FC', 'GPL201_DHF_2_log2FC', 'GPL201_DHF_3_log2FC']
GPL570 列数: 13 -> ['GPL570_DHF_1_log2FC', 'GPL570_DHF_2_log2FC', 'GPL570_DHF_3_log2FC', 'GPL570_DHF_4_log2FC', 'GPL570_DHF_5_log2FC', 'GPL570_DHF_6_log2FC', 'GPL570_DHF_7_log2FC', 'GPL570_DHF_8_log2FC', 'GPL570_DHF_9_log2FC', 'GPL570_DHF_10_log2FC', 'GPL570_DHF_11_log2FC', 'GPL570_DHF_12_log2FC', 'GPL570_DHF_13_log2FC']

重命名映射结果:
  GPL201_DHF_1_log2FC            -> GSE43777_DHFvsDF_1
  GPL201_DHF_2_log2FC            -> GSE43777_DHFvsDF_2
  GPL201_DHF_3_log2FC            -> GSE43777_DHFvsDF_3
  GPL570_DHF_1_log2FC            -> GSE43777_DHFvsDF_4
  GPL570_DHF_2_log2FC            -> GSE43777_DHFvsDF_5
  GPL570_DHF_3_log2FC            -> GSE43777_DHFvsDF_6
  GPL570_DHF_4_log2FC            -> GSE43777_DHFvsDF_7
  GPL570_DHF_5_log2FC            -> GSE4

In [10]:
"""
Step 2: 核心查询函数
============================
目标：
1. 写一个 query_gene() 函数：输入基因名 -> 返回该基因在127名患者中的原始log2FC值，
   并带上每个值对应的 数据集/严重程度/患者序号 标签
2. 支持两种视图模式：
   - "combined": DHF+DSS 合并成一个"Severe Dengue"大类，按数据集分组
   - "split":    DHF 和 DSS 分开，每个"数据集+严重程度"算一组
3. 空值处理：直接从原始表里取值后dropna，不填补
4. 用真实基因(STAT1, IFIT3)测试，打印结果验证正确性
5. 额外测试：不存在的基因、大小写不敏感搜索
"""

import pandas as pd
import numpy as np

GENE_COL = "Gene_Symbol"

# -------------------------------------------------
# 2.1 读取Step1保存的中间结果
# -------------------------------------------------
print("=" * 60)
print("STEP 2.1: 加载Step1处理好的数据")
print("=" * 60)

df = pd.read_pickle("df_processed.pkl")
col_meta = pd.read_pickle("col_meta.pkl")

print(f"表达矩阵形状: {df.shape}")
print(f"列元数据形状: {col_meta.shape}")
print(f"列元数据字段: {list(col_meta.columns)}")

# -------------------------------------------------
# 2.2 核心查询函数
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 2.2: 定义 query_gene() 核心函数")
print("=" * 60)


def query_gene(gene_symbol: str, mode: str = "combined"):
    """
    查询指定基因在所有患者中的log2FC值。

    Parameters
    ----------
    gene_symbol : str
        用户输入的基因名，大小写不敏感
    mode : str
        "combined" -> DHF+DSS合并，按数据集分组（7个数据集，7个箱子）
        "DHF"      -> 只看DHF vs DF，按数据集分组（只统计有DHF数据的数据集）
        "DSS"      -> 只看DSS vs DF，按数据集分组（只统计有DSS数据的数据集）

    Returns
    -------
    dict: 同之前结构
    """
    match = df[df[GENE_COL].astype(str).str.upper() == gene_symbol.strip().upper()]

    if len(match) == 0:
        return {"found": False, "gene_symbol": gene_symbol, "long_df": None,
                "group_summary": None, "total_valid_n": 0, "total_n": None}

    row = match.iloc[0]
    real_gene_name = row[GENE_COL]

    expr_cols = col_meta["column"].tolist()
    values = row[expr_cols]

    long_df = col_meta.copy()
    long_df["log2FC"] = long_df["column"].map(values)
    long_df_valid = long_df.dropna(subset=["log2FC"]).copy()

    # ---- 关键修改：按mode先过滤severity，再统一按dataset分组 ----
    if mode == "combined":
        filtered = long_df_valid.copy()
        total_n_possible = 127  # 全部患者都可能纳入
    elif mode in ("DHF", "DSS"):
        filtered = long_df_valid[long_df_valid["severity"] == mode].copy()
        # 该severity类型在原始数据里总共有多少患者(不管这个基因有没有检测到)
        total_n_possible = (col_meta["severity"] == mode).sum()
    else:
        raise ValueError("mode 必须是 'combined' / 'DHF' / 'DSS'")

    filtered["group"] = filtered["dataset"]  # 统一按数据集分组

    total_valid_n = len(filtered)

    group_summary = (
        filtered.groupby("group")["log2FC"]
        .agg(n="count", mean="mean", median="median", std="std")
        .reset_index()
        .sort_values("group")
        .reset_index(drop=True)
    )

    return {
        "found": True,
        "gene_symbol": real_gene_name,
        "long_df": filtered,
        "group_summary": group_summary,
        "total_valid_n": total_valid_n,
        "total_n": total_n_possible,   # 注意：DHF模式下是77，DSS模式下是50，combined是127
        "mode": mode,
    }

print("✅ query_gene() 函数定义完成")

# -------------------------------------------------
# 2.3 测试：STAT1 三种模式 (combined / DHF / DSS)
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 2.3: 测试查询 STAT1 (三种模式)")
print("=" * 60)

for mode in ["combined", "DHF", "DSS"]:
    result = query_gene("STAT1", mode=mode)
    print(f"--- mode = {mode} ---")
    print(f"找到基因: {result['found']}")
    print(f"基因名(原始大小写): {result['gene_symbol']}")
    print(f"有效值患者数: {result['total_valid_n']} / {result['total_n']}")
    print()
    print(f"按数据集分组统计 ({mode}):")
    print(result["group_summary"].to_string(index=False))
    print()

# -------------------------------------------------
# 2.4 验证：combined 模式的总患者数 应该等于 DHF模式 + DSS模式 的总数
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 2.4: 验证 combined = DHF + DSS")
print("=" * 60)

result_combined = query_gene("STAT1", mode="combined")
result_dhf = query_gene("STAT1", mode="DHF")
result_dss = query_gene("STAT1", mode="DSS")

n_combined = result_combined["total_valid_n"]
n_dhf = result_dhf["total_valid_n"]
n_dss = result_dss["total_valid_n"]

print(f"combined 有效患者数 = {n_combined}")
print(f"DHF 有效患者数      = {n_dhf}")
print(f"DSS 有效患者数      = {n_dss}")
print(f"DHF + DSS 之和      = {n_dhf + n_dss}")

assert n_combined == n_dhf + n_dss, "combined应该等于DHF+DSS之和！"
print("✅ combined = DHF + DSS，逻辑正确")

# -------------------------------------------------
# 2.5 测试：大小写不敏感 + 不存在的基因
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 2.5: 边界情况测试")
print("=" * 60)

r_lower = query_gene("stat1", mode="combined")
print(f"小写输入 'stat1' -> found={r_lower['found']}, 匹配到基因名={r_lower['gene_symbol']}")
assert r_lower["found"] and r_lower["gene_symbol"] == "STAT1"
print("✅ 大小写不敏感匹配正确")

for mode in ["combined", "DHF", "DSS"]:
    r_fake = query_gene("NOTAREALGENE123", mode=mode)
    print(f"[{mode}] 不存在的基因 -> found={r_fake['found']}")
    assert r_fake["found"] == False
print("✅ 三种模式下，不存在基因都正确返回 found=False")

# -------------------------------------------------
# 2.6 验证 DHF/DSS 模式下的数据集覆盖情况（呼应Step3发现的边界情况）
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 2.6: 验证 DHF/DSS 模式下涉及的数据集数量")
print("=" * 60)

datasets_dhf = sorted(result_dhf["group_summary"]["group"].unique())
datasets_dss = sorted(result_dss["group_summary"]["group"].unique())

print(f"DHF模式涉及的数据集 ({len(datasets_dhf)}个): {datasets_dhf}")
print(f"DSS模式涉及的数据集 ({len(datasets_dss)}个): {datasets_dss}")

# 根据Step1的dataset_summary，只有GSE17924/GSE25226/GSE38246/GSE94892同时有DHF和DSS
expected_dss = sorted(["GSE17924", "GSE25226", "GSE38246", "GSE94892"])
assert datasets_dss == expected_dss, "DSS模式的数据集覆盖不符合预期！"
print("✅ DSS模式正确只包含4个有DSS患者的数据集")

# -------------------------------------------------
# 2.7 再测试几个免疫相关基因，确认结果合理 (用combined模式看整体趋势)
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 2.7: 多基因测试 (IFIT3, CXCL10, ISG15, OAS1)")
print("=" * 60)

for g in ["IFIT3", "CXCL10", "ISG15", "OAS1"]:
    r = query_gene(g, mode="combined")
    if r["found"]:
        overall_mean = r["long_df"]["log2FC"].mean()
        print(f"{g:10s} | found=True  | valid_n={r['total_valid_n']:3d}/127 | overall_mean_log2FC={overall_mean:+.3f}")
    else:
        print(f"{g:10s} | found=False | 不在数据集中")

print()
print("✅ Step 2 全部测试通过")

STEP 2.1: 加载Step1处理好的数据
表达矩阵形状: (12746, 128)
列元数据形状: (127, 5)
列元数据字段: ['column', 'dataset', 'severity', 'control', 'patient_idx']

STEP 2.2: 定义 query_gene() 核心函数
✅ query_gene() 函数定义完成

STEP 2.3: 测试查询 STAT1 (三种模式)
--- mode = combined ---
找到基因: True
基因名(原始大小写): STAT1
有效值患者数: 127 / 127

按数据集分组统计 (combined):
   group  n      mean    median      std
GSE17924 32 -0.042471 -0.041625 0.058551
GSE18090 10  -0.63784 -0.611051 0.415404
GSE25226 14  0.495985  0.486342 0.920778
GSE38246 21 -0.242427 -0.271941 1.063659
GSE43777 16 -0.006192  0.010223 0.072616
GSE51808 10 -0.042787 -0.081379 0.091277
GSE94892 24 -1.261127 -1.218859 0.758697

--- mode = DHF ---
找到基因: True
基因名(原始大小写): STAT1
有效值患者数: 77 / 77

按数据集分组统计 (DHF):
   group  n      mean    median      std
GSE17924 13 -0.026856 -0.029498 0.071119
GSE18090 10  -0.63784 -0.611051 0.415404
GSE25226  6  0.402175  0.486342 0.936784
GSE38246 13 -0.204964 -0.271941 1.065025
GSE43777 16 -0.006192  0.010223 0.072616
GSE51808 10 -0.042787 -0.081379 0.0912

In [12]:
"""
Step 3: 可视化函数
============================
目标：
1. 写一个 plot_gene_boxplot() 函数：输入 query_gene() 的结果 -> 输出Plotly箱线图
2. 箱子标签要显示 n=xxx (真实有效患者数)
3. 支持 combined / split 两种模式（沿用Step2的分组逻辑）
4. 加 y=0 参考线，颜色按severity区分（DHF蓝色系，DSS红色系），方便一眼看出方向和严重程度
5. 用STAT1、IFIT3等基因测试，保存图片验证
"""

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 复用Step2的 query_gene() 函数，这里假设已经在同一个session/脚本里定义好了
# 如果是独立脚本运行，请把Step2的 query_gene() 函数定义也复制到这个文件里，
# 或者用 from step2_query_function import query_gene, df, col_meta 导入

print("=" * 60)
print("STEP 3.1: 定义 plot_gene_boxplot() 可视化函数")
print("=" * 60)


def plot_gene_boxplot(query_result: dict):
    if not query_result["found"]:
        return None

    gene_name = query_result["gene_symbol"]
    long_df = query_result["long_df"].copy()
    total_valid_n = query_result["total_valid_n"]
    total_n = query_result["total_n"]
    mode = query_result["mode"]

    if len(long_df) == 0:
        return None

    group_n = long_df.groupby("group").size().to_dict()
    long_df["group_label"] = long_df["group"].map(lambda g: f"{g} (n={group_n[g]})")

    order = sorted(long_df["group"].unique())
    order_labels = [f"{g} (n={group_n[g]})" for g in order]

    mode_display = {"combined": "Severe Dengue (DHF+DSS)", "DHF": "DHF", "DSS": "DSS"}[mode]
    color_map = {"combined": "#4C72B0", "DHF": "#4C72B0", "DSS": "#C44E52"}

    fig = px.box(
        long_df,
        x="group_label",
        y="log2FC",
        points="all",
        category_orders={"group_label": order_labels},
        title=f"{gene_name} — log2FC Distribution ({mode_display} vs DF)",
        labels={"group_label": "Dataset", "log2FC": "log2 Fold Change"},
        color_discrete_sequence=[color_map[mode]],
    )

    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.6)

    # -------- 关键新增：整体 n 作为醒目的图注(annotation)，独立于标题 --------
    fig.add_annotation(
        text=f"<b>Overall: n = {total_valid_n} / {total_n} patients analysed</b> "
             f"({total_valid_n/total_n*100:.0f}% detection rate)",
        xref="paper", yref="paper",
        x=0.5, y=1.08,          # 放在图的顶部、标题下方
        showarrow=False,
        font=dict(size=13, color="#333333"),
        align="center",
    )

    fig.update_layout(
        xaxis_tickangle=-30,
        template="plotly_white",
        showlegend=False,
        height=580,
        width=max(700, 120 * len(order)),
        margin=dict(t=100),  # 给顶部annotation留出空间
    )

    return fig


print("✅ plot_gene_boxplot() 函数定义完成")

# -------------------------------------------------
# 3.2 测试: STAT1 三种模式 (combined / DHF / DSS)
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 3.2: 测试绘图 - STAT1 (三种模式)")
print("=" * 60)

for mode in ["combined", "DHF", "DSS"]:
    result = query_gene("STAT1", mode=mode)
    fig = plot_gene_boxplot(result)
    fig.write_html(f"test_STAT1_{mode}.html")
    n_datasets = result["group_summary"]["group"].nunique()
    print(f"[{mode}] 已保存: test_STAT1_{mode}.html")
    print(f"[{mode}] valid_n = {result['total_valid_n']}/{result['total_n']}, 涉及数据集数 = {n_datasets}")
    print(result["group_summary"].to_string(index=False))
    print()

# -------------------------------------------------
# 3.3 测试: IFIT3 三种模式 (有缺失值的情况，重点检验空值处理)
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 3.3: 测试绘图 - IFIT3 (有缺失值的情况，三种模式)")
print("=" * 60)

for mode in ["combined", "DHF", "DSS"]:
    result = query_gene("IFIT3", mode=mode)
    fig = plot_gene_boxplot(result)
    fig.write_html(f"test_IFIT3_{mode}.html")
    n_datasets = result["group_summary"]["group"].nunique()
    print(f"[{mode}] 已保存: test_IFIT3_{mode}.html")
    print(f"[{mode}] valid_n = {result['total_valid_n']}/{result['total_n']}, 涉及数据集数 = {n_datasets}")
    print(result["group_summary"].to_string(index=False))
    print()

# -------------------------------------------------
# 3.4 关键验证: DSS模式下应该只剩4个数据集（GSE18090/GSE43777/GSE51808没有DSS患者）
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 3.4: 验证 DSS-only 模式下的数据集覆盖情况")
print("=" * 60)

result_dss = query_gene("STAT1", mode="DSS")
datasets_in_dss = sorted(result_dss["group_summary"]["group"].unique())
expected_dss_datasets = sorted(["GSE17924", "GSE25226", "GSE38246", "GSE94892"])

print(f"DSS模式下实际出现的数据集: {datasets_in_dss}")
print(f"预期应该出现的数据集(有DSS患者的4个): {expected_dss_datasets}")
assert datasets_in_dss == expected_dss_datasets, "DSS模式下的数据集覆盖不符合预期！"
print("✅ DSS-only模式正确排除了3个只有DHF患者的数据集")

# -------------------------------------------------
# 3.5 边界测试: 不存在的基因不应该报错
# -------------------------------------------------
print()
print("=" * 60)
print("STEP 3.5: 边界测试 - 不存在的基因")
print("=" * 60)

for mode in ["combined", "DHF", "DSS"]:
    result_fake = query_gene("NOTAREALGENE123", mode=mode)
    fig_fake = plot_gene_boxplot(result_fake)
    print(f"[{mode}] 不存在基因的绘图结果: {fig_fake}")
    assert fig_fake is None
print("✅ 三种模式下，不存在的基因都正确返回 None，不会报错崩溃")

print()
print("✅ Step 3 全部测试通过")

STEP 3.1: 定义 plot_gene_boxplot() 可视化函数
✅ plot_gene_boxplot() 函数定义完成

STEP 3.2: 测试绘图 - STAT1 (三种模式)
[combined] 已保存: test_STAT1_combined.html
[combined] valid_n = 127/127, 涉及数据集数 = 7
   group  n      mean    median      std
GSE17924 32 -0.042471 -0.041625 0.058551
GSE18090 10  -0.63784 -0.611051 0.415404
GSE25226 14  0.495985  0.486342 0.920778
GSE38246 21 -0.242427 -0.271941 1.063659
GSE43777 16 -0.006192  0.010223 0.072616
GSE51808 10 -0.042787 -0.081379 0.091277
GSE94892 24 -1.261127 -1.218859 0.758697

[DHF] 已保存: test_STAT1_DHF.html
[DHF] valid_n = 77/77, 涉及数据集数 = 7
   group  n      mean    median      std
GSE17924 13 -0.026856 -0.029498 0.071119
GSE18090 10  -0.63784 -0.611051 0.415404
GSE25226  6  0.402175  0.486342 0.936784
GSE38246 13 -0.204964 -0.271941 1.065025
GSE43777 16 -0.006192  0.010223 0.072616
GSE51808 10 -0.042787 -0.081379 0.091277
GSE94892  9 -1.081839  -0.84484 0.718888

[DSS] 已保存: test_STAT1_DSS.html
[DSS] valid_n = 50/50, 涉及数据集数 = 4
   group  n      mean    median

In [4]:
pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [kaleido]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
